# LF1–3 — correctness vs ground-truth

Sinh nhãn **độ hữu dụng** theo hướng *task-driven*: một ảnh phù hợp cho tác vụ khi
mô hình hạ nguồn của tác vụ đó xử lý **đúng** so với ground-truth. Đây là tín hiệu mạnh
nhất trong khung weak-supervision (xem `AGENTS.md`, `paper/Labeling_Plan.md`).

Bộ 3 nhãn:
- `1_maturity_evaluation` — GT = mức chín `dry/green/tender` (bộ Roboflow, YOLO .txt)
- `2_foliar_disease` — GT = lớp bệnh lá (Gray Leaf Spot, Leaf Rot)
- `3_trunk_crown_disease` — GT = lớp bệnh thân/ngọn (Stem Bleeding, Bud Rot, Bud Root Dropping)

Mô hình hạ nguồn được để dưới dạng **hàm giữ chỗ** (placeholder, mặc định trả `None`) nhằm cho phép notebook chạy được toàn trình trước khi tích hợp mô hình thực.

In [ ]:
# Cấu hình
from pathlib import Path
import csv
import numpy as np

ROOT = Path.cwd()
if not (ROOT/'Dataset').exists():
    for p in ROOT.parents:
        if (p/'Dataset').exists(): ROOT = p; break

TASKS = ["1_maturity_evaluation", "2_foliar_disease", "3_trunk_crown_disease"]
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

# Ánh xạ thư mục bệnh -> tác vụ (ground-truth lớp bệnh)
DISEASE_TASK = {
    "Gray Leaf Spot": "2_foliar_disease", "Leaf Rot": "2_foliar_disease",
    "Stem Bleeding": "3_trunk_crown_disease", "Bud Rot": "3_trunk_crown_disease",
    "Bud Root Dropping": "3_trunk_crown_disease",
}
# data.yaml Roboflow: names=['dry','green','tender'] -> class 0/1/2 = mức độ chín
YOLO_STAGE = {0: "dry", 1: "green", 2: "tender"}
print("ROOT =", ROOT)

## 1. Đọc ground-truth

In [ ]:
def read_yolo_stages(label_txt):
    """Mức độ chín (dry/green/tender) của các trái trong ảnh, theo class YOLO."""
    if not label_txt.exists(): return []
    out = []
    for line in label_txt.read_text().splitlines():
        if line.strip(): out.append(YOLO_STAGE.get(int(line.split()[0]), "?"))
    return out

def iter_roboflow(root):
    base = root/"Dataset"/"coconut-veirf-v5"
    for split in ("train", "valid", "test"):
        img_dir, lbl_dir = base/split/"images", base/split/"labels"
        if not img_dir.exists(): continue
        for img in sorted(img_dir.iterdir()):
            if img.suffix.lower() in IMG_EXTS:
                yield img, split, read_yolo_stages(lbl_dir/(img.stem + ".txt"))

def iter_disease(root):
    base = root/"Dataset"/"Coconut Tree Disease Dataset"
    if not base.exists(): return
    for folder in sorted(base.iterdir()):
        if not folder.is_dir(): continue
        gt_task = DISEASE_TASK.get(folder.name)
        for img in sorted(folder.rglob("*")):
            if img.suffix.lower() in IMG_EXTS:
                yield img, folder.name, gt_task

## 2. Mô hình hạ nguồn (hàm giữ chỗ — thay bằng mô hình thực)

Mặc định trả `None` → ảnh "chưa đánh giá được" (cột nhãn để trống), để notebook chạy được
trước khi có model thật.

In [ ]:
from dataclasses import dataclass
from typing import Callable

@dataclass
class DownstreamModels:
    predict_maturity: Callable = lambda p: None      # -> 'dry'/'green'/'tender'
    predict_foliar: Callable = lambda p: None        # -> bool: có phải bệnh lá?
    predict_trunk_crown: Callable = lambda p: None   # -> bool: có phải bệnh thân/ngọn?

## 3. Correctness -> nhãn hữu dụng

In [ ]:
def maturity_correct(pred_stage, gt_stages):
    """Đúng nếu mức chín dự đoán nằm trong tập mức thật của ảnh."""
    if pred_stage is None or not gt_stages: return None
    return int(pred_stage in gt_stages)

def cls_correct(pred_is_class, gt_is_class):
    """Đúng lớp bệnh -> hữu dụng. None/abstain nếu ảnh không thuộc tác vụ đó."""
    if not gt_is_class: return None
    if pred_is_class is None: return None
    return int(bool(pred_is_class))

## 4. Chạy & ghi manifest

In [ ]:
def build(root, models, out):
    rows = []
    # Roboflow -> độ chín
    for img, split, gt_stages in iter_roboflow(root):
        row = {"image_id": img.stem, "source": f"coconut-veirf-v5/{split}",
               "path": str(img.relative_to(root))}
        for t in TASKS: row[t] = np.nan
        row["1_maturity_evaluation"] = maturity_correct(models.predict_maturity(img), gt_stages)
        row["label_basis"] = "maturity=correctness(dry/green/tender)"
        rows.append(row)
    # Bệnh -> bệnh lá / thân-ngọn
    for img, folder, gt_task in iter_disease(root):
        row = {"image_id": img.stem, "source": f"disease/{folder}",
               "path": str(img.relative_to(root))}
        for t in TASKS: row[t] = np.nan
        row["2_foliar_disease"] = cls_correct(models.predict_foliar(img), gt_task == "2_foliar_disease")
        row["3_trunk_crown_disease"] = cls_correct(models.predict_trunk_crown(img), gt_task == "3_trunk_crown_disease")
        row["label_basis"] = f"disease-class={gt_task}"
        rows.append(row)
    out.parent.mkdir(parents=True, exist_ok=True)
    fields = ["image_id", "source", "path", *TASKS, "label_basis"]
    with out.open("w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=fields); w.writeheader()
        for r in rows:
            w.writerow({k: ("" if (isinstance(r.get(k), float) and np.isnan(r[k])) else r.get(k, "")) for k in fields})
    def col(t):
        vals = [r[t] for r in rows if r[t] is not None and not (isinstance(r[t], float) and np.isnan(r[t]))]
        return len(vals), sum(v for v in vals if v == 1)
    print(f"Đã ghi {out}  ({len(rows)} ảnh)")
    for t in TASKS:
        n, pos = col(t); print(f"  {t:24s}: {n:5d} / {pos}" + ("  <-- CHỜ model" if n == 0 else ""))

### Tích hợp mô hình thực rồi chạy

```python
models = DownstreamModels(
    predict_maturity    = lambda p: my_maturity_clf.predict(p),   # 'dry'/'green'/'tender'
    predict_foliar      = lambda p: my_leaf_clf.predict(p) in FOLIAR_CLASSES,
    predict_trunk_crown = lambda p: my_trunk_clf.predict(p) in TRUNK_CLASSES,
)
```

In [ ]:
models = DownstreamModels()      # hàm giữ chỗ: chưa tích hợp mô hình -> cột nhãn để trống
build(ROOT, models, ROOT/"labels"/"correctness_manifest.csv")